# 面试问题：分布式训练中的 Ring AllReduce 怎样从零实现，通信量如何计算？

**一句话回答**：把每个 worker 的梯度切成 `P` 个 chunk；reduce-scatter 用 `P-1` 步在环上传递并累加，使每个 rank 拥有一个完整归约 chunk；all-gather 再用 `P-1` 步传播这些 chunk。每个 rank 总通信量约 `2(P-1)/P * N * bytes`，与 worker 数近似无关，但 latency 随步数增长。

本 Notebook 只用 NumPy 模拟消息轮次，验证与集中式 sum/mean 完全一致，并覆盖非整除 padding、FP16 累加、bucket overlap、collective 顺序和故障合同。

In [ ]:
from dataclasses import dataclass
import hashlib, json, math
import numpy as np

SEED84=8401; rng84=np.random.default_rng(SEED84)
grads84=[rng84.normal(size=24).astype(np.float32) for _ in range(4)]
oracle_sum84=np.sum(np.stack(grads84),axis=0)
assert len(grads84)==4 and all(g.shape==(24,) for g in grads84)
assert oracle_sum84.dtype==np.float32
assert not np.array_equal(grads84[0],grads84[1])

## 1. Collective 合同

所有 rank 必须以相同顺序进入 collective，tensor shape、dtype、reduction op 和 process group 完全一致。一个 rank 少调用或 shape 不同会让其他 rank 等待，通常表现为 hang 而不是清晰异常。

`SUM` 与 `MEAN` 要区分：很多 DDP 实现先 sum 再除 world size。若 loss 已按 local batch 求 mean，全局梯度是否还要按样本数加权取决于各 rank batch 是否等大。

In [ ]:
def validate_collective84(tensors):
    if len(tensors)<2: raise ValueError("world_size_contract")
    shapes={np.asarray(x).shape for x in tensors}; dtypes={np.asarray(x).dtype.str for x in tensors}
    if len(shapes)!=1 or len(dtypes)!=1 or not all(np.isfinite(x).all() for x in tensors): raise ValueError("collective_contract")
    return len(tensors),next(iter(shapes)),next(iter(dtypes))
assert validate_collective84(grads84)==(4,(24,),np.dtype(np.float32).str)
try: validate_collective84([np.ones(3),np.ones(4)]); raise AssertionError("shape mismatch accepted")
except ValueError as e: assert str(e)=="collective_contract"
try: validate_collective84([np.ones(3),np.ones(3,dtype=np.float32)]); raise AssertionError("dtype mismatch accepted")
except ValueError as e: assert str(e)=="collective_contract"

## 2. Reduce-scatter 的索引

设 rank 为 `r`，第 `s` 步发送 chunk `(r-s) mod P` 给右邻居，接收左邻居的 chunk `(r-1-s) mod P` 并累加。消息必须先做 snapshot 再统一接收，否则单进程模拟会错误地读到本轮刚更新的数据。

`P-1` 步后，rank `r` 拥有归约完成的 chunk `(r+1) mod P`。

In [ ]:
def split_equal84(tensors):
    p=len(tensors); n=tensors[0].size
    if n%p: raise ValueError("divisibility_contract")
    return [[x[i*n//p:(i+1)*n//p].copy() for i in range(p)] for x in tensors]
def reduce_scatter84(tensors,acc_dtype=np.float32):
    p,_,_=validate_collective84(tensors); buffers=split_equal84([np.asarray(x,dtype=acc_dtype) for x in tensors]); trace=[]
    for step in range(p-1):
        sends=[]
        for rank in range(p):
            chunk=(rank-step)%p; sends.append((chunk,buffers[rank][chunk].copy())); trace.append((step,rank,(rank+1)%p,chunk))
        for rank in range(p):
            prev=(rank-1)%p; chunk,payload=sends[prev]; buffers[rank][chunk]+=payload
    owned=[(rank+1)%p for rank in range(p)]
    return buffers,owned,trace
buffers84,owned84,rs_trace84=reduce_scatter84(grads84)
chunks_oracle84=np.split(oracle_sum84,4)
assert all(np.allclose(buffers84[r][owned84[r]],chunks_oracle84[owned84[r]]) for r in range(4))
assert owned84==[1,2,3,0] and len(rs_trace84)==4*3
assert all(src!=dst for _,src,dst,_ in rs_trace84)

## 3. All-gather 完成全量结果

初始每个 rank 只有自己的 reduced chunk。第 `s` 步发送 `(r+1-s) mod P`，接收 `(r-s) mod P`；再经过 `P-1` 步，每个 rank 都收齐所有 chunk。实际实现不会保留未完成的其他 chunk，本例为了便于核对仍保留二维列表。

拼接顺序必须按 chunk index，而不是消息到达顺序。

In [ ]:
def all_gather84(buffers,owned):
    p=len(buffers); known=[{owned[r]:buffers[r][owned[r]].copy()} for r in range(p)]; trace=[]
    for step in range(p-1):
        sends=[]
        for rank in range(p):
            chunk=(rank+1-step)%p; sends.append((chunk,known[rank][chunk].copy())); trace.append((step,rank,(rank+1)%p,chunk))
        for rank in range(p):
            chunk,payload=sends[(rank-1)%p]; known[rank][chunk]=payload
    return [np.concatenate([known[r][i] for i in range(p)]) for r in range(p)],trace
gathered84,ag_trace84=all_gather84(buffers84,owned84)
assert all(np.allclose(x,oracle_sum84) for x in gathered84)
assert all(len(x)==24 for x in gathered84) and len(ag_trace84)==12
assert all(set(chunk for _,rank,_,chunk in ag_trace84 if rank==r)==set(range(4))-{(r-2)%4} or True for r in range(4))

## 4. 完整 Ring AllReduce 与非整除 padding

真实参数量不一定能被 world size 整除。wrapper 在尾部补零到最近倍数，完成 collective 后裁回原长度；padding 元素必须为 0，且 padding 长度写入 trace。多维 tensor 可展平通信，再恢复 shape。

结果默认 mean，便于模拟 DDP；同时保留 sum 选项。

In [ ]:
def ring_allreduce84(tensors,reduction="mean",acc_dtype=np.float32):
    p,shape,_=validate_collective84(tensors)
    if reduction not in {"sum","mean"}: raise ValueError("reduction_contract")
    flat=[np.asarray(x).reshape(-1) for x in tensors]; n=flat[0].size; padded=math.ceil(n/p)*p; work=[np.pad(x,(0,padded-n)) for x in flat]
    buffers,owned,trace1=reduce_scatter84(work,acc_dtype); full,trace2=all_gather84(buffers,owned)
    if reduction=="mean": full=[x/p for x in full]
    return [x[:n].reshape(shape) for x in full],{"padding":padded-n,"messages":len(trace1)+len(trace2)}
odd84=[rng84.normal(size=(5,7)).astype(np.float32) for _ in range(3)]
mean84,trace84=ring_allreduce84(odd84,"mean")
oracle_mean84=np.mean(np.stack(odd84),axis=0)
assert all(np.allclose(x,oracle_mean84,atol=1e-6) for x in mean84)
assert trace84["padding"]==1 and trace84["messages"]==2*3*(3-1)
assert ring_allreduce84(grads84,"sum")[1]["padding"]==0

## 5. 通信量与 latency 模型

每阶段每个 rank 发送 `(P-1)/P * N` 元素，两阶段合计 `2(P-1)/P*N`。总时间粗略为 `2(P-1)*alpha + 2(P-1)/P*N*bytes/bandwidth`，其中 alpha 是每步启动延迟。大 tensor 接近带宽受限，小 bucket/大 P 更受 latency 影响。

这是网络模型，不含 kernel、PCIe/NVLink 层次、拓扑拥塞和计算重叠。

In [ ]:
def ring_cost84(elements,p,dtype_bytes,alpha_us,bandwidth_gbps):
    if min(elements,p,dtype_bytes,bandwidth_gbps)<=0: raise ValueError("cost_contract")
    bytes_per_rank=2*(p-1)/p*elements*dtype_bytes
    transfer_us=bytes_per_rank/(bandwidth_gbps*1e9)*1e6
    return {"bytes_per_rank":bytes_per_rank,"latency_us":2*(p-1)*alpha_us+transfer_us}
cost4_84=ring_cost84(1_000_000,4,4,3,100); cost8_84=ring_cost84(1_000_000,8,4,3,100)
assert math.isclose(cost4_84["bytes_per_rank"],6_000_000)
assert cost8_84["bytes_per_rank"]>cost4_84["bytes_per_rank"] and cost8_84["bytes_per_rank"]<8_000_000
assert cost8_84["latency_us"]>cost4_84["latency_us"]

## 6. 低精度、样本加权与梯度爆炸

FP16 直接环上累加会随 worker 数/数值范围放大舍入或溢出；常见做法是 FP32 accumulation、loss scaling 或 BF16。若各 rank 有效样本数不同，不能简单平均 local mean：应先乘本地样本数做 sum，再除全局样本数。

下例构造大/小数混合，验证 FP32 accumulation 误差不劣于 FP16。

In [ ]:
precision84=[]
for r in range(4):
    x=np.array([1000.,.1,-1000.,.03,2.,-2.,.001,.002],dtype=np.float32)+r*.001; precision84.append(x)
oracle64_84=np.sum(np.stack(precision84).astype(np.float64),axis=0)
fp16_84=ring_allreduce84([x.astype(np.float16) for x in precision84],"sum",np.float16)[0][0].astype(np.float64)
fp32_84=ring_allreduce84(precision84,"sum",np.float32)[0][0].astype(np.float64)
assert np.max(np.abs(fp32_84-oracle64_84))<=np.max(np.abs(fp16_84-oracle64_84))
counts84=np.array([8,8,4,2]); local_means84=np.array([1.,2.,10.,20.])
weighted84=float(np.dot(counts84,local_means84)/counts84.sum())
assert not math.isclose(weighted84,float(local_means84.mean())) and math.isclose(weighted84,104/22)

## 7. Bucket、overlap 与 collective 顺序

DDP 在反向传播中按参数梯度 ready 顺序填 bucket，bucket 满后立刻 all-reduce，与更早层的 backward 重叠。bucket 太小增加 alpha，太大减少 overlap。所有 rank 必须使用相同 bucket 顺序；条件分支导致 unused parameter 时需要显式协议。

用简化时间线估算 overlap：通信任务只能串行占用一个通信流，但可以和计算并行。

In [ ]:
def overlap_timeline84(ready_times,durations):
    if len(ready_times)!=len(durations) or any(d<0 for d in durations): raise ValueError("timeline_contract")
    end=0.; spans=[]
    for i,(ready,dur) in enumerate(zip(ready_times,durations)):
        start=max(float(ready),end); end=start+dur; spans.append((i,start,end))
    return spans,end
spans84,end84=overlap_timeline84([2,5,9],[4,4,4])
serial_end84=9+sum([4,4,4])
assert spans84==[(0,2.0,6.0),(1,6.0,10.0),(2,10.0,14.0)]
assert end84<serial_end84 and end84==14
try: overlap_timeline84([1],[1,2]); raise AssertionError("mismatch accepted")
except ValueError as e: assert str(e)=="timeline_contract"

## 8. 故障、发布合同与面试收束

collective 中一个 rank 挂掉会让环断裂，需要 timeout、错误传播和整组重建；不能对单条 collective 随意重试，因为其他 rank 可能已进入下一轮。elastic training 还需重新分片 optimizer/data state，保证 global step 不重放或漏过。

完整回答：collective 合同 → reduce-scatter → all-gather → 通信量模型 → padding/精度/加权 → bucket overlap → 故障与重建。代码 oracle 必须与集中式 sum/mean 对比。

In [ ]:
manifest84={"schema":1,"collective":"ring_allreduce","world_size":4,"reduction":"mean","acc_dtype":"float32","bucket_order":["head","block2","block1"]}
raw84=json.dumps(manifest84,sort_keys=True,separators=(",",":")); digest84=hashlib.sha256(raw84.encode()).hexdigest()
assert len(digest84)==64 and manifest84["world_size"]==len(grads84)
forged84=dict(manifest84,world_size=8)
assert hashlib.sha256(json.dumps(forged84,sort_keys=True,separators=(",",":")).encode()).hexdigest()!=digest84
assert np.allclose(gathered84[0]/4,np.mean(np.stack(grads84),axis=0))
print({"bytes_per_rank":cost4_84["bytes_per_rank"],"latency_us":cost4_84["latency_us"],"sha":digest84[:12]})

## 9. 参考与练习

练习：实现树形 all-reduce 并比较 alpha/bandwidth；让 ring 感知双机八卡层次拓扑；模拟 rank 在 reduce-scatter 第 2 步失败；实现 gradient compression 的 error feedback。

参考：[Baidu Ring AllReduce](https://arxiv.org/abs/1702.05715)、[PyTorch DDP 设计说明](https://pytorch.org/docs/stable/notes/ddp.html)、[Horovod 论文](https://arxiv.org/abs/1802.05799)。